In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

import argparse
import logging
import os
import sys

import numpy as np
import torch
import torch.nn as nn
from torch import optim
from tqdm import tqdm
import wandb
import pickle

import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch import Tensor

from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader, random_split


imgs_dir = "/kaggle/input/cloud-masking-dataset/content/train/data/"
masks_dir = "/kaggle/input/cloud-masking-dataset/content/train/masks/"

In [ ]:
# import tifffile as tiff
# import matplotlib.pyplot as plt

# file_path = "/kaggle/input/cloud-masking-dataset/content/train/data/103642.tif"
# img_array = tiff.imread(file_path)

# print("Image shape:", img_array.shape)

# rgb_channels = img_array[..., :3]

# # Determine the maximum value in the RGB channels
# max_value = np.max(rgb_channels)
# print("Maximum pixel value before normalization:", max_value)

# rgb_normalized = rgb_channels.astype(np.float32) / max_value

# plt.figure(figsize=(6, 6))
# plt.imshow(rgb_normalized)
# plt.title("Normalized RGB Image (0-1)")
# plt.axis("off")
# plt.show()


# mask_path = "/kaggle/input/cloud-masking-dataset/content/train/masks/103642.tif"
# mask_array = tiff.imread(mask_path)

# print("Min pixel",np.min(mask_array))
# print(mask_array.shape)
# print(mask_array)
# mask_normalized = mask_array.astype(np.float32) 
# plt.figure(figsize=(6, 6))
# plt.imshow(mask_normalized)
# plt.title("Mask image")
# plt.axis("off")
# plt.show()


In [ ]:
def dice_loss(input: Tensor, target: Tensor, epsilon: float = 1e-6):
    # Average of Dice coefficient for all batches, or for a single mask
    assert input.size() == target.size()

    sum_dim = (-1, -2, -3)

    inter = 2 * (input * target).sum(dim=sum_dim)
    sets_sum = input.sum(dim=sum_dim) + target.sum(dim=sum_dim)
    sets_sum = torch.where(sets_sum == 0, inter, sets_sum)

    dice = (inter + epsilon) / (sets_sum + epsilon)
    ############ should change this ##############
    return 1 - dice.mean()



In [ ]:
class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""

    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downscaling with maxpool then double conv"""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upscaling then double conv"""

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels , in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)


    def forward(self, x1, x2):
        x1 = self.up(x1)
        # input is CHW
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        # if you have padding issues, see
        # https://github.com/HaiyongJiang/U-Net-Pytorch-Unstructured-Buggy/commit/0e854509c2cea854e247a9c615f175f76fbb2e3a
        # https://github.com/xiaopeng-liao/Pytorch-UNet/commit/8ebac70e633bac59fc22bb5195e513d5832fb3bd
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=True):
        super(UNet, self).__init__()
        
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = OutConv(64, n_classes)

    def forward(self, x):
            x1 = self.inc(x)
            x2 = self.down1(x1)
            x3 = self.down2(x2)
            x4 = self.down3(x3)
            x5 = self.down4(x4)
            x = self.up1(x5, x4)
            x = self.up2(x, x3)
            x = self.up3(x, x2)
            x = self.up4(x, x1)
            logits = self.outc(x)
            return logits
    

In [ ]:
from os.path import splitext
from os import listdir
import numpy as np
from glob import glob
import torch
from torch.utils.data import Dataset
import logging
from PIL import Image
import tifffile as tiff


class BasicDataset(Dataset):
    def __init__(self, imgs_dir, masks_dir, scale=1, mask_suffix=''):
        self.imgs_dir = imgs_dir
        self.masks_dir = masks_dir
        self.scale = scale
        assert 0 < scale <= 1, 'Scale must be between 0 and 1'

        self.ids = [splitext(file)[0] for file in listdir(imgs_dir)
                    if not file.startswith('.')]
        print(f'Creating dataset with {len(self.ids)} examples')

    def __len__(self):
        return len(self.ids)

    @classmethod
    def preprocess(cls, tif_img, scale):
        # w, h = (tif_img[0],tif_img[1])
        # newW, newH = int(scale * w), int(scale * h)
        # assert newW > 0 and newH > 0, 'Scale is too small'
        # tif_img = tif_img.resize((newW, newH))

        img_nd = np.array(tif_img)

        if len(img_nd.shape) == 2:
            img_nd = np.expand_dims(img_nd, axis=2)

        # HWC to CHW
        img_trans = img_nd.transpose((2, 0, 1))
        if img_trans.max() > 1:
            ############# reconsider this division ###############
            img_trans = img_trans / 65535

        return img_trans

    def __getitem__(self, i):
        idx = self.ids[i]

        
        mask_file = glob(self.masks_dir + idx + ".tif")
        img_file = glob(self.imgs_dir+idx+".tif")

        assert len(mask_file) == 1, \
            f'Either no mask or multiple masks found for the ID {idx}: {mask_file}'
        assert len(img_file) == 1, \
            f'Either no image or multiple images found for the ID {idx}: {img_file}'
        mask = tiff.imread(mask_file[0])
        img = tiff.imread(img_file[0])

        assert img.size/4 == mask.size, \
            f'Image and mask {idx} should be the same size, but are {img.shape} and {mask.shape}'

        img = self.preprocess(img, self.scale)
        mask = self.preprocess(mask, self.scale)

        return {
            'image': torch.from_numpy(img).type(torch.FloatTensor),
            'mask': torch.from_numpy(mask).type(torch.FloatTensor)
        }


class CarvanaDataset(BasicDataset):
    def __init__(self, imgs_dir, masks_dir, scale=1):
        super().__init__(imgs_dir, masks_dir, scale, mask_suffix='_mask')

In [ ]:
imgs_dir = "/kaggle/input/cloud-masking-dataset/content/train/data/"
masks_dir = "/kaggle/input/cloud-masking-dataset/content/train/masks/"

data_set = BasicDataset( imgs_dir, masks_dir, scale=1)
data_set[5]


In [ ]:
dir_img = "/kaggle/input/cloud-masking-dataset/content/train/data/"
dir_mask = "/kaggle/input/cloud-masking-dataset/content/train/masks/"
dir_checkpoint = '/kaggle/working/'

In [ ]:

@torch.inference_mode()
def evaluate(net, dataloader, device, amp):
    net.eval()
    num_val_batches = len(dataloader)
    dice_score = 0

    # iterate over the validation set
    with torch.autocast(device.type if device.type != 'mps' else 'cpu', enabled=amp):
        for batch in tqdm(dataloader, total=num_val_batches, desc='Validation round', unit='batch', leave=False):
            image, mask_true = batch['image'], batch['mask']

            # move images and labels to correct device and type
            image = image.to(device=device)
            mask_true = mask_true.to(device=device)

            # predict the mask
            mask_pred = net(image)

            assert mask_true.min() >= 0 and mask_true.max() <= 1, 'True mask indices should be in [0, 1]'
            mask_pred = (F.sigmoid(mask_pred) > 0.5).float()
            # compute the Dice score
            dice_score += dice_loss(mask_pred, mask_true)

    net.train()
    return 1- dice_score / max(num_val_batches, 1)

In [ ]:


def train_model(
        model,
        device,
        epochs: int = 5,
        batch_size: int = 1,
        learning_rate: float = 1e-5,
        val_percent: float = 0.1,
        save_checkpoint: bool = True,
        img_scale: float = 0.5,
        amp: bool = False,
        weight_decay: float = 1e-8,
        momentum: float = 0.999,
        gradient_clipping: float = 1.0,
):
    dataset = BasicDataset(dir_img, dir_mask, img_scale)
    n_val = int(len(dataset) * val_percent)
    n_train = len(dataset) - n_val
    train_set, val_set = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(0))
    loader_args = dict(batch_size=batch_size, num_workers=os.cpu_count(), pin_memory=True)
    train_loader = DataLoader(train_set, shuffle=True, **loader_args)
    val_loader = DataLoader(val_set, shuffle=False, drop_last=True, **loader_args)
    # experiment = wandb.init(project='U-Net', resume='allow', anonymous='must')
    # experiment.config.update(
    #     dict(epochs=epochs, batch_size=batch_size, learning_rate=learning_rate,
    #          val_percent=val_percent, save_checkpoint=save_checkpoint, img_scale=img_scale, amp=amp)
    # )

    optimizer = optim.RMSprop(model.parameters(),
                              lr=learning_rate, weight_decay=weight_decay, momentum=momentum, foreach=True)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=5)  
    grad_scaler = torch.cuda.amp.GradScaler(enabled=amp)
    criterion = nn.BCEWithLogitsLoss()
    global_step = 0

    
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0
        with tqdm(total=n_train, desc=f'Epoch {epoch}/{epochs}', unit='img') as pbar:
            for batch in train_loader:
                images, true_masks = batch['image'], batch['mask']
                assert images.shape[1] == model.n_channels, \
                    f'Network has been defined with {model.n_channels} input channels, ' \
                    f'but loaded images have {images.shape[1]} channels. Please check that ' \
                    'the images are loaded correctly.'

                images = images.to(device=device)
                true_masks = true_masks.to(device=device)
                with torch.autocast(device.type if device.type != 'mps' else 'cpu', enabled=amp):
                    masks_pred = model(images)
                    loss = criterion(masks_pred.squeeze(1), true_masks.squeeze(1).float())
                    loss += dice_loss(F.sigmoid(masks_pred.squeeze(1)), true_masks.squeeze(1).float())

                optimizer.zero_grad(set_to_none=True)
                grad_scaler.scale(loss).backward()
                grad_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clipping)
                grad_scaler.step(optimizer)
                grad_scaler.update()

                torch.cuda.empty_cache()
                
                pbar.update(images.shape[0])
                global_step += 1
                epoch_loss += loss.item()
                
                # experiment.log({
                #     'train loss': loss.item(),
                #     'step': global_step,
                #     'epoch': epoch
                # })
                pbar.set_postfix(**{'loss (batch)': loss.item()})

                division_step = (n_train // (5 * batch_size))
                if division_step > 0:
                    if global_step % division_step == 0:
                        val_score = evaluate(model, val_loader, device, amp)
                        scheduler.step(val_score)
                        print('Validation Dice score: {}'.format(val_score))
                # if(global_step == 3):
                #     break

        if save_checkpoint:
            pickle.dump(model , open(f'model{epoch}.pk1' , 'wb'))
            print(f'Checkpoint {epoch} saved!')


In [ ]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.cuda.empty_cache()

model = UNet(4, 1)
model = model.to(memory_format=torch.channels_last)
model.to(device=device)
torch.cuda.memory_summary(device=None, abbreviated=False)

print(device)
train_model(
            model=model,
            epochs=10,
            batch_size=10,
            learning_rate=0.001,
            device=device,
            img_scale=1,
            val_percent=0.1,
            amp=False
        )

In [ ]:
def predict_img(net,
                full_img,
                device,
                scale_factor=1,
                out_threshold=0.5):
    net.eval()
    img = full_img
    img = img.unsqueeze(0)
    img = img.to(device=device)

    with torch.no_grad():
        output = net(img).cpu()
        ########### This is used when scaled only #########
        # output = F.interpolate(output, (full_img.size[1], full_img.size[0]), mode='bilinear')
        mask = torch.sigmoid(output) > out_threshold
    print("Output mask shape: ",mask.shape)
    return mask[0].long().squeeze().numpy()


In [ ]:
img = data_set[1]['image']
print(img.shape)
mask_array = data_set[1]['mask'].squeeze().numpy()
pred_mask_array = predict_img(model,img,device)

new_img = img.numpy().transpose(1, 2, 0)


rgb_channels = new_img[..., :3]


plt.figure(figsize=(6, 6))
plt.imshow(rgb_channels)
plt.title("Normalized RGB Image (0-1)")
plt.axis("off")
plt.show()


plt.figure(figsize=(6, 6))
plt.imshow(mask_array)
plt.title("Mask image")
plt.axis("off")
plt.show()
print(mask_array)
plt.figure(figsize=(6, 6))
plt.imshow(pred_mask_array)
plt.title("Predicted mask image")
plt.axis("off")
plt.show()

print(pred_mask_array)
